# VGG16 + probly auf einem Bild

Sehr einfaches Beispiel: Wir nehmen das bekannte Bildnetz **VGG16**, lassen es ein Foto klassifizieren und fragen mit **probly**, **wie sicher** es sich dabei ist.

| Abschnitt | Inhalt |
|---|---|
| 1 | Imports |
| 2 | VGG16 laden |
| 3 | Bild laden und vorbereiten |
| 4 | Normale Vorhersage (ohne probly) |
| 5 | probly: Dropout-Modell und Unsicherheit |
| 6 | Vergleich: Foto gegen verrauschtes Foto gegen reines Rauschen |
| 7 | Zusammenfassung |

## 1. Imports

In [5]:
%pip install torch torchvision pillow matplotlib

Note: you may need to restart the kernel to use updated packages.


In [4]:
import warnings
warnings.filterwarnings("ignore")

import os, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from torchvision.models import vgg16, VGG16_Weights

from probly.method import dropout
from probly.representer import representer
from probly.quantification import quantify

torch.manual_seed(0)

- `torchvision` enthält VGG16 und die zugehörigen Vorverarbeitungsschritte
- `PIL` lädt das Bild, `matplotlib` zeigt es an
- `dropout`, `representer`, `quantify` sind wieder die drei probly-Bausteine: Modell umbauen, viele Vorhersagen holen, Unsicherheit berechnen

## 2. VGG16 laden
Wir benutzen ein **bereits trainiertes** VGG16 (auf ImageNet, 1000 Klassen wie „Hund", „Katze", „Auto"). Wir müssen es also nicht selbst trainieren.

In [6]:
gewichte = VGG16_Weights.IMAGENET1K_V1
vgg = vgg16(weights=gewichte)
vgg.eval()                                  # Vorhersagemodus

klassen = gewichte.meta["categories"]       # die 1000 Klassennamen
print("Anzahl Klassen:", len(klassen))

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /Users/tristanfuchs/.cache/torch/hub/checkpoints/vgg16-397923af.pth


URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1020)>

- `vgg16(weights=...)` lädt das Netz mit fertigen Gewichten. Beim ersten Mal wird es heruntergeladen (etwa 500 MB).
- `vgg.eval()` schaltet in den Vorhersagemodus.
- VGG16 besteht aus 13 Faltungsschichten und 3 vollverbundenen Schichten. Es gibt für ein Bild 1000 Zahlen aus (Logits), eine pro Klasse.
- In den vollverbundenen Schichten hat VGG16 schon **Dropout-Schichten** (`nn.Dropout`). Das nutzen wir in Abschnitt 5.

## 3. Bild laden und vorbereiten
Du kannst ein eigenes Bild benutzen: lege es als `bild.jpg` neben das Notebook. Gibt es keins, laden wir ein Beispielfoto (einen Hund) herunter.

In [ ]:
pfad = "bild.jpg"
if not os.path.exists(pfad):
    urllib.request.urlretrieve("https://github.com/pytorch/hub/raw/master/images/dog.jpg", pfad)

bild = Image.open(pfad).convert("RGB")
vorverarbeitung = gewichte.transforms()          # Größe 224x224 + Normalisierung
x = vorverarbeitung(bild).unsqueeze(0)           # Form (1, 3, 224, 224)

print("Form der Eingabe:", tuple(x.shape))
plt.imshow(bild); plt.axis("off"); plt.title("Unser Bild"); plt.show()

VGG16 erwartet Bilder in einem bestimmten Format:
- `transforms()` skaliert das Bild auf 224 × 224 Pixel und normalisiert die Farben genau so, wie das Netz damals trainiert wurde
- `unsqueeze(0)` fügt eine „Bild-Achse" hinzu: Form (1, 3, 224, 224) = 1 Bild, 3 Farbkanäle, 224 × 224 Pixel

## 4. Normale Vorhersage (ohne probly)
Zuerst machen wir es so, wie man es normalerweise tut: Bild rein, Klasse raus.

In [ ]:
with torch.no_grad():
    logits = vgg(x)
wahrsch = torch.softmax(logits, dim=1)[0]

top5 = torch.topk(wahrsch, 5)
for p, i in zip(top5.values, top5.indices):
    print(f"{klassen[i]:30s} {p.item():.3f}")

- `softmax` macht aus den 1000 Logits Wahrscheinlichkeiten, die zusammen 1 ergeben
- `topk(5)` nimmt die 5 wahrscheinlichsten Klassen

Das Netz nennt eine Klasse mit einer Wahrscheinlichkeit. **Wie verlässlich** diese Zahl ist, sagt es nicht. Genau das macht probly möglich.

## 5. probly: Dropout-Modell und Unsicherheit
Wir bauen VGG16 mit probly zu einem **MC-Dropout-Modell** um. Dann geht dasselbe Bild mehrfach durch das Netz, jedes Mal mit anderem Dropout, und wir sehen, wie stark die Antworten schwanken.

In [ ]:
mc  = dropout(vgg, p=0.2, predictor_type="logit_classifier")
rep = representer(mc, num_samples=20)         # 20 Durchläufe pro Bild

with torch.no_grad():
    sample = rep.represent(x)

mittel = sample.sample_mean().probabilities.detach()[0]
unsicher = quantify(sample).total.detach().numpy()[0]

top5 = torch.topk(mittel, 5)
print("Mittlere Vorhersage über 20 Durchläufe:")
for p, i in zip(top5.values, top5.indices):
    print(f"  {klassen[i]:30s} {p.item():.3f}")
print(f"\nGesamt-Unsicherheit: {unsicher:.5f}")

Was hier passiert:
- `dropout(vgg, p=0.2, ...)` kopiert das Netz und fügt Dropout-Schichten ein. Das Original `vgg` bleibt unverändert.
- `representer(mc, num_samples=20)` schickt jedes Bild 20-mal durch das Netz. Dropout ist dabei automatisch **angeschaltet**.
- `sample_mean()` mittelt die 20 Antworten. Das ist die Vorhersage.
- `quantify(sample).total` misst, **wie stark die 20 Antworten schwanken**. Das ist die Gesamt-Unsicherheit: klein = die Durchläufe sind sich einig, groß = sie widersprechen sich.

Die Zahl allein sagt noch wenig. Sie wird erst im nächsten Abschnitt aussagekräftig, wenn wir sie mit anderen Bildern vergleichen.

## 6. Vergleich: Foto, verrauschtes Foto, reines Rauschen
Wir füttern das Netz mit drei Bildern, die immer „unbekannter" werden, und schauen, ob die Unsicherheit steigt.

In [ ]:
bilder = {
    "Originalfoto":      x,
    "Stark verrauscht":  x + 1.5 * torch.randn_like(x),
    "Reines Rauschen":   torch.randn(1, 3, 224, 224),
}

mittel_n = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1)
std_n    = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1)
def zum_anzeigen(t):
    # Normalisierung rückgängig machen, damit man das Bild sehen kann
    a = t[0].numpy() * std_n + mittel_n
    return np.clip(a.transpose(1, 2, 0), 0, 1)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
print(f"{'Bild':18s} {'Klasse':28s} {'P':>6s}  {'Unsicherheit':>12s}")
for ax, (name, bild_t) in zip(axes, bilder.items()):
    with torch.no_grad():
        s = rep.represent(bild_t)
    p = s.sample_mean().probabilities.detach()[0]
    u = quantify(s).total.detach().numpy()[0]
    top = int(p.argmax())
    print(f"{name:18s} {klassen[top]:28s} {p[top].item():6.3f}  {u:12.5f}")
    ax.imshow(zum_anzeigen(bild_t)); ax.axis("off")
    ax.set_title(f"{name}\n{klassen[top]} ({p[top].item():.2f})\nUnsicherheit {u:.4f}")
plt.tight_layout(); plt.show()

**So liest man das Ergebnis:**
- **Originalfoto:** Das Netz kennt solche Bilder. Die Unsicherheit sollte klein sein.
- **Stark verrauscht:** Das Bild ist beschädigt. Das Netz wird unsicherer, die Unsicherheit sollte steigen.
- **Reines Rauschen:** Das Netz hat so etwas nie gesehen. Es nennt trotzdem irgendeine Klasse, aber die Unsicherheit sollte hier am höchsten sein.

Das ist der Kern von probly: Auch wenn das Netz bei Unsinn selbstbewusst eine Klasse ausgibt, zeigt die Unsicherheit, dass man dem Ergebnis nicht trauen sollte. Die genauen Zahlen hängen vom Bild ab, achte vor allem auf die **Reihenfolge** der Unsicherheiten.

## 7. Zusammenfassung
- **VGG16** ist ein fertiges, vortrainiertes Bildnetz. Es gibt eine Klasse und eine Wahrscheinlichkeit aus, aber keine Aussage darüber, wie verlässlich das ist.
- **probly** baut daraus mit `dropout(...)` ein Modell, das dasselbe Bild mehrfach mit anderem Dropout bewertet.
- Die **Streuung** dieser Bewertungen (`quantify(...).total`) ist die Unsicherheit: klein bei bekannten Bildern, größer bei beschädigten oder fremden.

**Nächste Schritte zum Ausprobieren:** eigenes Bild als `bild.jpg` speichern, `num_samples` erhöhen (z. B. 50), `p` ändern (z. B. 0.5) oder ein Bild nehmen, das keine der 1000 Klassen zeigt (z. B. ein Diagramm).